## The Three Editions

| Edition | What It Includes |
|---|---|
| **Core** | Streaming ingest only |
| **Pro** | Core + CDC (change data capture: you can update and delete rows in the target table) |
| **Advanced** | Pro + Expectations (a rule that says "this row must satisfy X to be considered valid") |

## The Three Actions

| Action | SQL | Python | Behavior |
|---|---|---|---|
| **WARN** | `ON VIOLATION WARN` | `@dlt.expect` | Keep the row, log a warning |
| **DROP ROW** | `ON VIOLATION DROP ROW` | `@dlt.expect_or_drop` | Silently remove the row |
| **FAIL UPDATE** | `ON VIOLATION FAIL UPDATE` | `@dlt.expect_or_fail` | Stop the entire pipeline |

In [0]:
CREATE OR REFRESH STREAMING TABLE clickstream_raw
COMMENT "Raw Wikipedia clickstream data."
AS
SELECT
  curr_title AS current_page_title,
  prev_title AS previous_page_title,
  CAST(n AS INT) AS click_count,
  type AS link_type
FROM STREAM read_files(
  "/databricks-datasets/wikipedia-datasets/data-001/clickstream/raw-uncompressed-json",
  format => 'json'
);

CREATE OR REFRESH STREAMING TABLE clickstream_clean(
  -- Rule 1: Drop rows with missing page title
  CONSTRAINT valid_current_page  
    EXPECT (current_page_title IS NOT NULL) 
    ON VIOLATION DROP ROW,

  -- Rule 2: Fail the pipeline if click_count is invalid
  CONSTRAINT valid_click_count 
    EXPECT (click_count > 0) 
    ON VIOLATION FAIL UPDATE,

  -- Rule 3: Warn (but keep) rows with very short titles
  CONSTRAINT warn_short_page 
    EXPECT (LENGTH(current_page_title) > 3) 
    --ON VIOLATION WARN (warn is the default behavior when no action is specified.)
)
COMMENT "Cleaned clickstream with three expectation types."
AS
SELECT
  current_page_title,
  previous_page_title,
  click_count,
  link_type
FROM STREAM(live.clickstream_raw);

CREATE OR REFRESH MATERIALIZED VIEW top_clickstream_pages
COMMENT "Aggregated click counts."
AS
SELECT
  current_page_title,
  SUM(click_count) AS total_clicks,
  COUNT(*) AS link_types_count
FROM live.clickstream_clean
GROUP BY current_page_title;

**DLT makes SCD much easier — one command instead of two MERGEs.**

In [0]:
-- Step 1: Create the target streaming table
CREATE OR REFRESH STREAMING TABLE customers_clean;

-- Step 2: Apply changes (THIS is CDC)
APPLY CHANGES INTO live.customers_clean
FROM STREAM(live.customers_raw)
KEYS (customer_id)
SEQUENCE BY updated_at
APPLY AS DELETE WHEN operation = 'DELETE'
COLUMNS * EXCEPT (operation, updated_at)
STORED AS SCD TYPE 1;

In [0]:
CREATE OR REFRESH STREAMING TABLE dim_customer;

APPLY CHANGES INTO live.dim_customer
FROM STREAM(live.customer_changes)
KEYS (customer_id)
SEQUENCE BY updated_at
STORED AS SCD TYPE 2;

In [0]:
%python
#python
import dlt
from pyspark.sql.functions import col

dlt.create_streaming_table("dim_customer")

dlt.apply_changes(
    target="dim_customer",
    source="customer_changes",
    keys=["customer_id"],
    sequence_by=col("updated_at"),
    stored_as_scd_type=2
)

# SCD without DLT

## SCD Type 1 — Overwrite

**Behavior:** Update the row in place. No history.

### Example

**Before:**

| customer_id | name  | city |
| ----------- | ----- | ---- |
| 1           | Alice | NYC  |

**After Alice moves to LA:**

| customer_id | name  | city |
| ----------- | ----- | ---- |
| 1           | Alice | LA   |

**Old value "NYC" is gone.** If you didn't back it up, it's lost.

### Use Case

**When you don't need history** — e.g., fixing typos, correcting wrong data.

### Delta Implementation

```sql
MERGE INTO dim_customer AS t
USING updates AS s
ON t.customer_id = s.customer_id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;
```

This is standard upsert — no history.

---

## SCD Type 2 — Full History (The Important One)

**Behavior:** Add a **new row** for each change. Keep all old versions.

### Example

**Before:**

| customer_id | name  | city | __start_date | __end_date | __is_current |
| ----------- | ----- | ---- | ------------ | ---------- | ------------ |
| 1           | Alice | NYC  | 2020-01-01   | 9999-12-31 | true         |

**After Alice moves to LA:**

| customer_id | name  | city | __start_date | __end_date | __is_current |
| ----------- | ----- | ---- | ------------ | ---------- | ------------ |
| 1           | Alice | NYC  | 2020-01-01   | 2024-06-15 | false        |
| 1           | Alice | LA   | 2024-06-15   | 9999-12-31 | true         |

**Both versions exist.** You can query "where did Alice live in 2022?" → NYC.

### The Three Magic Columns

| Column                           | Meaning                                      |
| -------------------------------- | -------------------------------------------- |
| `__start_date` (or `valid_from`) | When this version became active              |
| `__end_date` (or `valid_to`)     | When this version ended (or 9999 if current) |
| `__is_current`                   | Boolean — is this the active version?        |

**Alternative convention:** `valid_from` and `valid_to` (no `__is_current` flag).

`DLT creates them automatically when you use APPLY CHANGES INTO`

### Use Case

**When history matters:**

* Auditing — "what was the customer's address when they placed this order?"
* Analytics — "how many customers lived in NYC in 2022?"
* Compliance — "prove we had their consent at the time"

### Delta Implementation — Two Steps

**Step 1: Mark old versions as inactive**

```sql
MERGE INTO dim_customer AS t
USING updates AS s
ON t.customer_id = s.customer_id
   AND t.__is_current = true
WHEN MATCHED
 AND (t.name != s.name OR t.city != s.city)
THEN
  UPDATE SET
    t.__is_current = false,
    t.__end_date = current_date();
```

**Step 2: Insert new versions**

```sql
INSERT INTO dim_customer
SELECT
    s.customer_id,
    s.name,
    s.city,
    current_date() AS __start_date,
    DATE('9999-12-31') AS __end_date,
    true AS __is_current
FROM updates s
LEFT JOIN dim_customer t
  ON t.customer_id = s.customer_id
 AND t.__is_current = true
WHERE t.customer_id IS NULL
   OR t.name != s.name
   OR t.city != s.city;
```

**Note:** New inserts (never-seen customers) also handled by the second query.

**Explanation**
```LEFT JOIN dim_customer t
  ON t.customer_id = s.customer_id AND t.__is_current = true```

  **The join condition is:** `customer_id = 1 AND __is_current = true`

  So `t` is NULL for this row. The `LEFT JOIN` produces:

| customer_id | name  | city | t.customer_id | t.name | t.city |
| ----------- | ----- | ---- | ------------- | ------ | ------ |
| 1           | Alice | LA   | NULL          | NULL   | NULL   |


```
use <=> in place of != for null saftey
for ex: null =null
it returns null which is not true
null<=>null 
will return True
```

in place of **t.name != s.name** write **NOT(t.name <=> s.name)**